# Home Credit Default Risk：最速ベースライン

このノートブックの目的は、表形式の二値分類データを読み込み、最小限の検査、Quick EDA、前処理、5-fold CV、submission作成までを一度通すことです。

最初は上から **submission作成まで** を実行します。Adversarial validationはその後に実行し、train/test間の分布差を確認します。


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Set up

再現性を保つための乱数シード、CVの分割数、目的変数、ID列をまとめて設定します。別のコンペで使う場合は、まずこのセルの定数を変更します。


In [ ]:
from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

SEED = 42
N_SPLITS = 5
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

pd.set_option("display.max_columns", 122)
pd.set_option("display.width", 140)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_train.csv")
test = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_test.csv")

print("train:",train.shape,"test:",test.shape)
display(train.head())

# Dataの確認

学習前に、目的変数とID列の存在、IDの一意性、train/test間のID重複を確認します。ここで失敗した場合は、モデル作成へ進まずデータ定義を見直します。


In [ ]:
assert TARGET in train.columns
assert TARGET not in test.columns
assert ID_COL in train.columns and ID_COL in test.columns
assert train[ID_COL].is_unique
assert test[ID_COL].is_unique
assert set(train[ID_COL]).isdisjoint(set(test[ID_COL]))

print("target values:", sorted(train[TARGET].unique().tolist()))
print("train SK_ID_CURR:", train[ID_COL].min(),"->",train[ID_COL].max())
print("test SK_ID_CURR:", test[ID_COL].min(),"->",test[ID_COL].max())
print("data contract : OK!")

# Quick EDA

モデル作成前のsanity checkとして、列型、目的変数の偏り、欠損率、train/test間の欠損率差を確認します。ここではbaselineの設計に必要な情報だけを短時間で把握します。


In [ ]:
print("=== dtype ===")
display(train.dtypes.to_frame("dtype").T)

print("=== target balance ===")
target_rate = train[TARGET].mean()
print(f"TARGET = 1 rate: {target_rate:.3f}")
display(train[TARGET].value_counts(normalize=True).rename("ratio").to_frame())

In [ ]:
def basic_profile(df, name):
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_rate": df.isna().mean(),
        "nunique": df.nunique(dropna=False),
    })
    out["dataset"] = name
    return out.sort_values(["missing_rate","nunique"], ascending=[False,False])

print("=== train profile ===")
display(basic_profile(train, "train").head())

print("=== test prifile ===")
display(basic_profile(test, "test").head())

In [ ]:
common_cols = [c for c in test.columns if c in train.columns]
missing_compare = pd.DataFrame({
    "train_missing": train[common_cols].isna().mean(),
    "test_missing": test[common_cols].isna().mean(),
})
missing_compare["abs_diff"] = (missing_compare["train_missing"] - missing_compare["test_missing"]).abs()

display(missing_compare.sort_values("abs_diff",ascending=False).head(10))

# ベースライン用の特徴量定義

Quick EDAの確認結果を踏まえ、ID列と目的変数を除外し、数値列とカテゴリ列を機械的に分けます。


In [ ]:
from pandas.api.types import is_numeric_dtype
# ID列と目的変数を除き、モデルに入力する候補を作る
BASE_FEATURES = [c for c in train.columns if c not in [ID_COL, TARGET]]
# 値の種類が5以上ある数値列は連続値として扱う
BASE_NUM = [c for c in BASE_FEATURES if is_numeric_dtype(train[c]) and train[c].nunique() >= 5]
# 文字列列と低カーディナリティの数値列はカテゴリ変数として扱う
BASE_CAT = [c for c in BASE_FEATURES if c not in BASE_NUM]

# Standard Baseline Modelをつくる

数値列は中央値補完、カテゴリ列は最頻値補完とOne-Hot Encodingを行い、XGBoostで二値分類します。前処理をPipelineに含めることで、各foldの学習データだけで前処理がfitされます。


In [ ]:
def make_baseline_model(seed=SEED):
    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), BASE_NUM),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), BASE_CAT)
    ])

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.04,
        min_child_weight=0.2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=seed,
        n_jobs=2,
        verbosity=0,
    )

    return Pipeline([
        ("prep", preprocess),
        ("model", xgb),
    ])

baseline_model = make_baseline_model()
baseline_model

In [ ]:
y = train[TARGET].astype(int)

majority_pred = np.repeat(int(y.mean() >= 0.5), len(y))

print(f"majority accuracy : {accuracy_score(y, majority_pred):.4f}")

# 素朴な5-fold CVに通してみる

StratifiedKFoldで目的変数の比率を保ちながら評価します。各foldの検証予測からOOF AUCを計算し、test予測は5モデルの平均を使います。


In [ ]:
def run_cv(model, X, y, X_test, splitter, groups=None, label="cv", save_prefix=None):
    oof = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_rows = []

    split_iter = splitter.split(X, y, groups) if groups is not None else splitter.split(X, y)

    for fold, (tr_idx, va_idx) in enumerate(split_iter):
        m = clone(model)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])

        va_prob = m.predict_proba(X.iloc[va_idx])[:, 1]
        te_prob = m.predict_proba(X_test)[:, 1]

        oof[va_idx] = va_prob
        test_pred += te_prob / splitter.get_n_splits()

        fold_auc = roc_auc_score(y.iloc[va_idx], va_prob)
        fold_rows.append({
            "cv": label,
            "fold": fold,
            "n_train": len(tr_idx),
            "n_valid": len(va_idx),
            "valid_target_rate": y.iloc[va_idx].mean(),
            "auc": fold_auc,
        })

    fold_df = pd.DataFrame(fold_rows)
    
    overall_auc = roc_auc_score(y, oof)

    print(f"[{label}] OOF AUC = {overall_auc:.5f}")
    print(f"fold mean = {fold_df['auc'].mean():.5f}, std = {fold_df['auc'].std(ddof=0):.5f}")
    display(fold_df)

    if save_prefix is not None:
        out_dir = Path("artifacts")
        out_dir.mkdir(exist_ok=True)
        np.save(out_dir / f"{save_prefix}_oof.npy", oof)
        np.save(out_dir / f"{save_prefix}_test.npy", test_pred)
        pd.DataFrame({
            ID_COL: train[ID_COL],
            TARGET: y,
            "oof_prob": oof,
        }).to_csv(out_dir / f"{save_prefix}_oof.csv", index=False)
        print("saved to", out_dir.resolve())

    return {
        "label": label,
        "oof": oof,
        "test_pred": test_pred,
        "fold_df": fold_df,
        "oof_auc": overall_auc,
    }

In [ ]:
X = train[BASE_FEATURES].copy()
X_test = test[BASE_FEATURES].copy()

naive_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

result_naive = run_cv(
    model=baseline_model,
    X=X,
    y=y,
    X_test=X_test,
    splitter=naive_cv,
    label="StratifiedKFold(seed=42)",
    save_prefix="baseline_v0_stratified",
)

In [ ]:
def fold_diagnostics(df, splitter, y, groups=None):
    rows = []
    split_iter = splitter.split(df, y, groups) if groups is not None else splitter.split(df, y)

    for fold, (_, va_idx) in enumerate(split_iter):
        va = df.iloc[va_idx]
        rows.append({
            "fold": fold,
            "n" : len(va),
            "target_rate": y.iloc[va_idx].mean(),
        })
    return pd.DataFrame(rows)

naive_diag = fold_diagnostics(train, naive_cv, y)
display(naive_diag)
print("column-wise std across folds")
display(naive_diag.drop(columns="fold").std().to_frame("std").T)

# とりあえずsubmissionまでやる

CVで作成したtest予測を提出形式に変換します。このセルまでを最初に実行し、動くbaselineとsubmissionを早めに確保します。


In [ ]:
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: (result_naive["test_pred"])
})

submission.to_csv("submission.csv",index=False)
display(submission.head())
print("saved: submission.csv")

# Submission作成後の追加診断

以下のAdversarial validationは最初のsubmissionには必須ではありません。baselineを確保してから実行し、train/test間の分布差を確認します。


## Adversarial validation

trainを0、testを1とする分類問題を作り、両者をどの程度見分けられるかを確認します。AUCが0.5に近ければ分布は似ており、高い場合はtrain/test間の分布差を疑います。これは目的変数を予測するモデルの性能評価ではなく、分布差の診断です。


In [ ]:
adv = pd.concat([
    train[BASE_FEATURES].assign(is_test=0),
    test[BASE_FEATURES].assign(is_test=1),
],ignore_index=True)

adv_preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), BASE_NUM),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), BASE_CAT)
])

adv_model = Pipeline([
    ("prep", adv_preprocess),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
])

adv_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
adv_auc = cross_val_score(
    adv_model,
    adv[BASE_FEATURES],
    adv["is_test"],
    scoring="roc_auc",
    cv=adv_cv,
)

print("adversarial AUC by fold: ", np.round(adv_auc,4))
print(f"mean={adv_auc.mean():.4f}, std={adv_auc.std():.4f}")